# TRABALHO PRÁTICO 1 - Inteligência Artificial
**DOCENTE:** Professora Érica Carvalho
**TEMA:** Segmentação de Clientes de Cartão de Crédito com Aprendizagem não Supervisionada.

---

## 1. Definição do Problema e Contextualização
As operadoras de cartões de crédito recolhem continuamente dados transacionais dos utilizadores. Compreender perfis de consumo e de pagamento permite desenhar limites de crédito ajustados ao risco, propor campanhas comerciais direcionadas e mitigar a inadimplência. Dado que não existem rótulos prévios para o perfil dos clientes, este problema enquadra-se na **aprendizagem não supervisionada (agrupamento/clustering)**.

## 2. Objetivo da Modelagem
Identificar agrupamentos (*clusters*) naturais de clientes com base em métricas comportamentais (saldos, frequência de compras, adiantamentos e histórico de liquidação de faturas) obtidas num período de 6 meses.

## 3. Descrição do Conjunto de Dados
* **Fonte:** *Credit Card Dataset for Clustering* (`CC GENERAL.csv`).
* **Instâncias:** 8.950 utilizadores.
* **Atributos:** 18 atributos comportamentais (incluindo `BALANCE`, `PURCHASES`, `CREDIT_LIMIT`, `PAYMENTS`) e 1 atributo de identificação (`CUST_ID`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#CONFIGURAÇÕES VISUAIS PARA OS GRÁFICOS
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 5)

#CARREGAMENTO DO DATASET
df = pd.read_csv('CC GENERAL.csv')

print(f"Dimensão inicial do conjunto de dados: {df.shape}")
df.head()


## 2. Eliminação Manual de Atributos
O atributo `CUST_ID` representa apenas a chave primária de identificação do cliente no sistema bancário. Por ser um identificador categórico único para cada linha, não contém padrão estático generalizável e causaria distorções no cálculo de distâncias. Desta forma, a sua remoção manual é essencial.

In [ ]:
#ELIMINAÇÃO DO ATRIBUTO IRRELEVANTE CUST_ID
df_tratado = df.drop(columns=['CUST_ID'])

print(f"Dimensão após eliminação: {df_tratado.shape}")
print("Atributos restantes:", df_tratado.columns.tolist()[:5], "...")

## 3. Limpeza de Dados
Antes da aplicação de transformações matemáticas, é indispensável identificar e tratar valores faltantes (`NaN`). Algoritmos de redução de dimensionalidade e de cálculo de distâncias não toleram valores ausentes. Para preservar o tamanho da amostra sem introduzir o viés da média (sensível a valores discrepantes), imputa-se a **mediana** nos atributos contínuos afetados.

In [ ]:
#INSPEÇÃO DE VALORES NULOS
nulos_iniciais = df_tratado.isnull().sum()
print("Valores ausentes encontrados:")
print(nulos_iniciais[nulos_iniciais > 0])

#IMPUTAÇÃO PELA MEDIANA DOS ATRIBUTOS CREDIT_LIMIT E MINIMUM PAYMENTS
df_tratado['CREDIT_LIMIT'] = df_tratado['CREDIT_LIMIT'].fillna(df_tratado['CREDIT_LIMIT'].median())
df_tratado['MINIMUM_PAYMENTS'] = df_tratado['MINIMUM_PAYMENTS'].fillna(df_tratado['MINIMUM_PAYMENTS'].median())

print(f"\nTotal de valores ausentes após a imputação: {df_tratado.isnull().sum().sum()}")

## 4. Amostragem de Dados
Para demonstrar o impacto do custo computacional em conjuntos de grande escala ou para validações rápidas de protótipos, extrai-se uma amostra aleatória representativa de 30% dos dados via amostragem aleatória simples.

In [ ]:
#AMOSTRAGEM DE 30% COM SEMENTE DEFINIDA PARA REPRODUTIBILIDADE
df_amostra = df_tratado.sample(frac=0.3, random_state=42)

print(f"Dimensão da amostra representativa (30%): {df_amostra.shape}")
df_amostra.head(3)

## 5. Transformação de Dados: Padronização
As variáveis financeiras possuem magnitudes numéricas distintas: enquanto `CREDIT_LIMIT`  e `BALANCE` atingem ordens de dezenas de milhares, atributos como `PURCHASES_FREQUENCY` variam apenas entre 0 e 1. Caso os dados não sejam padronizados, os atributos de maior escala dominariam arbitrariamente as medidas de dispersão. A aplicação de `StandardScaler` ajusta cada atributo para média zero e desvio-padrão unitário.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#AJUSTE E TRANSFORMAÇÃO SOBRE TODO O CONJUNTO LIMPO
dados_escalados = scaler.fit_transform(df_tratado)

df_scaled = pd.DataFrame(dados_escalados, columns=df_tratado.columns)

print("Exemplo das variáveis após padronização (média ~ 0, desvio-padrão ~ 1):")
df_scaled.head(3)

## 6. Redução da Dimensionalidade (PCA)
O espaço vetorial atual possui 17 dimensões contínuas. A Análise de Componentes Principais (PCA) projeta esses dados num número reduzido de variáveis latentes ortogonais (componentes principais), capturando a direção de máxima variância global para mitigar redundâncias e possibilitar a visualização bidimensional do espaço amostral.

In [ ]:
from sklearn.decomposition import PCA

#REDUÇÃO PARA 2 COMPONENTES PRINCIPAIS
pca = PCA(n_components=2)
dados_pca = pca.fit_transform(df_scaled)

df_pca = pd.DataFrame(dados_pca, columns=['PC1', 'PC2'])

var_exp = pca.explained_variance_ratio_
print(f"Variância explicada por PC1: {var_exp[0]*100:.2f}%")
print(f"Variância explicada por PC2: {var_exp[1]*100:.2f}%")
print(f"Variância explicada total retida: {sum(var_exp)*100:.2f}%")

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df_pca['PC1'], df_pca['PC2'], alpha=0.35, color='#2b5c8f', edgecolors='none', s=20)
plt.title('Dispersão do Espaço Amostral após PCA (2 Componentes)', fontsize=13, weight='bold')
plt.xlabel(f'Componente Principal 1 ({var_exp[0]*100:.1f}% da variância)')
plt.ylabel(f'Componente Principal 2 ({var_exp[1]*100:.1f}% da variância)')
plt.tight_layout()
plt.show()

## 7. Conclusões e Próximos passos
* **Limpeza e Consistência:** A base passou de 18 atributos para 17 preditores numéricos sem ruídos de identificação nem valores omissos.
* **Escalabilidade:** A normalização z-score garantiu que todos os atributos contribuam com pesos proporcionais nas distâncias euclidianas.
* **Redução Dimensional:** A extração dos componentes principais sintetizou o espaço multidimensional e permitiu inspecionar a densidade e o comportamento dos pontos no plano.
* **Continuidade:** Os dados processados em `df_scaled` e `df_pca` encontram-se prontos para a definição do número ótimo de clusters (Método do Cotovelo e Silhueta) e execução do algoritmo K-Means na entrega final.